# PhysioNet summary characteristics

## Recommended repository

PhysioNet is recommended by:

- Scientific Data: https://www.nature.com/sdata/policies/repositories
- PLOS Biology: https://journals.plos.org/plosbiology/s/recommended-repositories
- NUS: https://libguides.nus.edu.sg/rdm/selected_dr
- NeurIPS: https://neurips.cc/Conferences/2023/CallForDatasetsBenchmarks
- Springer Nature: https://www.springernature.com/gp/authors/research-data-policy/health-sciences-repositories/12327108
- PLOS One: https://journals.plos.org/plosone/s/recommended-repositories#loc-biomedical-sciences
- Elsevier: https://admin1.journals.elsevier.com/

## Datathons

List of datathons:  
https://docs.google.com/document/d/19x5sJFIsbJYAlTC6yegyDio_OhEIRV9Bz0GV57WpQi0/

## Setup

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from tableone import TableOne

In [ ]:
# path to datasets
base_path = os.path.join("..", "data", "physionet")

In [ ]:
# Custom function to parse the datetime
def parse_publish_date(date_str):
    try:
        # Try the format with microseconds first
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S.%f%z")
    except ValueError:
        # If that fails, try the format without microseconds
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S%z")

## Load data

In [ ]:
# iterate folder
for file in os.listdir(base_path):
    print(file)

In [ ]:
users = pd.read_csv(os.path.join(base_path, "users.csv"), low_memory=False)

# Convert date columns to date type
users['join_date'] = pd.to_datetime(users['join_date'], format="%Y-%m-%d")
# users = users.with_columns(pl.col("join_date").str.strptime(pl.Date, "%Y-%m-%d"))

users.head(3)

In [ ]:
projects = pd.read_csv(os.path.join(base_path, "projects.csv"), low_memory=False)

# Convert date columns to date type
# projects = projects.with_columns(pl.col("publish_date").str.to_datetime("%Y-%m-%d %H:%M:%S%.f%z", strict=False))
# projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')
# projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')
projects['publish_date'] = projects['publish_date'].apply(parse_publish_date).dt.date

# projects[projects.publish_date == pd.NaT]
projects.head()

## Summary statistics

In [ ]:
def safe_len(x):
    """
    calculate number of items in a cell
    """
    if isinstance(x, list):
        return len(x)
    elif isinstance(x, str):  # If it's a string, return the length of the string
        return x.count(',')
    else:
        return 0  # Return 0 for NaNs or other types (like floats)

In [ ]:
projects['number_authors'] = projects['author_ids'].apply(safe_len)

In [ ]:
# Add publication year
projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')
projects['publication_year'] = projects['publish_date'].dt.year

# Add number of authors
projects['number_authors'] = projects['author_ids'].apply(safe_len)

# Consider adding legacy column (i.e. pre september 2019)

# Harmonize DUAs
harmonize = {"Korea Credentialed Health Data Agreement-1-0-0": "Credentialed",
             "Medical AI Foundations Checkpoints: Additional Terms of Service": "Credentialed",
             "Bridge2AI Voice Registered Access Agreement":"Credentialed",
             "PhysioNet Contributor Review Health Data Use Agreement 1.5.0": "Contributor review",
             "PhysioNet Credentialed Health Data Use Agreement 1.5.0": "Credentialed",
             "PhysioNet Restricted Health Data Use Agreement 1.5.0": "Restricted",
             np.nan: "Open"}

# Apply the dictionary to the column using map
projects['access_type'] = projects['data_use_agreement'].map(harmonize)

# Convert Storage size (Mb) to Storage size (Gb)
projects['storage_size_gb'] = projects['storage_size_mb'] / 1000

In [ ]:
projects['data_use_agreement'].unique()

In [ ]:
# Should we limit to latest versions only?
projects = projects[projects['is_latest_version'] == True]

In [ ]:
# Define the bins (5-year intervals) and labels
# bins = list(range(1998, 2027, 7))
bins = list(range(1998, 2027, 4))
labels = [f'{start}-{end-1}' for start, end in zip(bins[:-1], bins[1:])]
print(labels)

In [ ]:
# Use pd.cut to group the years into buckets
projects['publication_year_group'] = pd.cut(projects['publication_year'],
                                            bins=bins, labels=labels, right=False)

In [ ]:
projects.columns

In [ ]:
columns = ["resource_type", "access_type", "storage_size_gb",
           "number_authors", "publication_year_group"]

groupby = "resource_type"
# groupby = "access_type"
categorical = ["access_type", "resource_type", "publication_year_group", "number_authors"].remove(groupby)

rename= {"resource_type": "Resource_type",
         "access_type": "Access type",
         "signed_dua_count": "Signed DUAs",
         "storage_size_gb": "Storage size (Gb)",
         "number_authors":"Number of authors",
         "publication_year_group":"Publication year"}

order = {"access_type": ["Open", "Restricted", "Credentialed", "Contributor review"],
         "resource_type": ["Database", "Software", "Challenge", "Model"]}

t1 = TableOne(projects, columns=columns, categorical=categorical, groupby=groupby,
              rename=rename, order=order, missing=False)
t1

In [ ]:
print(t1.tabulate(tablefmt="latex"))

## Metrics

Extract simple metrics from the user and project data.

### Number of new users over time

In [ ]:
# Convert 'join_date' to datetime if it's not already
users['join_date'] = pd.to_datetime(users['join_date'], errors='coerce')

# Extract the year from 'join_date' and create a new column 'year'
users['year'] = users['join_date'].dt.year

# Group by 'year' and count the number of users per year
users_per_year = users.groupby('year').agg(count=('user_id', 'size')).reset_index()

users_per_year.head(5)

In [ ]:
# Create the bar plot using Seaborn
plt.figure(figsize=(8, 6), dpi=300)

# Plotting with Seaborn
ax = sns.barplot(x="year", y="count", data=users_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("New Registered Users", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("New Registered Users Per Year", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks for better readability
plt.xticks(rotation=45, ha='right')

# Ensure y-ticks are integers and do not overlap
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Apply a tight layout to ensure everything fits well
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('new_users_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


In [ ]:
# Calculate the cumulative sum of new users per year
users_per_year['cumulative_count'] = users_per_year['count'].cumsum()

# Create the bar plot using Seaborn
plt.figure(figsize=(8, 6), dpi=300)

# Plotting the cumulative data with Seaborn
ax = sns.barplot(x="year", y="cumulative_count", data=users_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("Cumulative Registered Users", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("Cumulative Registered Users Over Years", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks for better readability
plt.xticks(rotation=45, ha='right')

# Ensure y-ticks are integers and do not overlap
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Apply a tight layout to ensure everything fits well
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('cumulative_users_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

### Number of credentialed users

In [ ]:
# Convert 'join_date' to datetime if it's not already
users['join_date'] = pd.to_datetime(users['join_date'], errors='coerce')

# Filter the DataFrame for credentialed users using .loc[]
credentialed_users = users.loc[users['credentialing_status'] == "Credentialed"].copy()

# Extract the year from 'join_date' and create a new column 'year' using .loc[]
credentialed_users.loc[:, 'year'] = credentialed_users['join_date'].dt.year

# Group by 'year' and count the number of users per year
credentialed_users_per_year = credentialed_users.groupby('year').agg(count=('user_id', 'size')).reset_index()

credentialed_users_per_year.head(3)

In [ ]:
# Create the bar plot using Seaborn
plt.figure(figsize=(8, 6), dpi=300)

# Plotting with Seaborn
ax = sns.barplot(x="year", y="count", data=credentialed_users_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("New Credentialed Users", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("Number of New Credentialed Users Per Year", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks for better readability
plt.xticks(rotation=45, ha='right')

# Ensure y-ticks are integers and do not overlap
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Apply a tight layout to ensure everything fits well
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('new_credentialed_users_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
# Calculate the cumulative sum of new users per year
credentialed_users_per_year['cumulative_count'] = credentialed_users_per_year['count'].cumsum()

# Create the bar plot using Seaborn
plt.figure(figsize=(8, 6), dpi=300)

# Plotting the cumulative data with Seaborn
ax = sns.barplot(x="year", y="cumulative_count", data=credentialed_users_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("Cumulative Credentialed Users", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("Cumulative Credentialed Users Over Years", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks for better readability
plt.xticks(rotation=45, ha='right')

# Ensure y-ticks are integers and do not overlap
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Apply a tight layout to ensure everything fits well
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('cumulative_credentialed_users_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

### Number of published projects over time

In [ ]:
# Convert 'publish_date' to datetime if it's not already
projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')

# Extract the year from 'publish_date' and create a new column 'year'
projects['year'] = projects['publish_date'].dt.year

# Group by 'year' and count the number of projects per year
projects_per_year = projects.groupby('year').agg(count=('project_id', 'size')).reset_index()

projects_per_year.head(3)

In [ ]:
# Create the bar plot using Seaborn
plt.figure(figsize=(8, 4), dpi=300)

# Use Seaborn's barplot
ax = sns.barplot(x="year", y="count", data=projects_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("Publications", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("New Published Projects Per Year", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks
plt.xticks(rotation=45, ha='right')  # This replaces set_xticklabels and correctly rotates the labels

# Ensure y-ticks do not overlap by setting a more appropriate scale
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # Ensures only integer labels are used

# Tight layout for better spacing
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('new_published_projects_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


In [ ]:
# cumulative projects
projects_per_year['cumulative_count'] = projects_per_year['count'].cumsum()

# Create the bar plot using Seaborn
plt.figure(figsize=(8, 4), dpi=300)

# Use Seaborn's barplot
ax = sns.barplot(x="year", y="cumulative_count", data=projects_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("Publications", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("Cumulative Published Projects", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks
plt.xticks(rotation=45, ha='right')  # This replaces set_xticklabels and correctly rotates the labels

# Ensure y-ticks do not overlap by setting a more appropriate scale
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # Ensures only integer labels are used

# Tight layout for better spacing
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('cumulative_published_projects_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
projects_per_year

### Number of projects submitted by year since 2020

Need to add submitted projects to the system

### OTHERS

- Proportion of projects accepted
- Breakdown by project type?
- Proportion of conference/journal papers that use PhysioNet data/software
- Volume of data that we are hosting
- Number of users signing DUA
- Average daily visits
- Average monthly users (visits)
- New Google Scholar articles citing
- Number of courses
- Number of patents
- Indication that what we are doing is useful
- Stanford paper on shaky foundations (PhysioNet supports a whole field of research)
- Number of datathons/workshops [Chrystinne]
- International reach (users and contributors)
- Who is contributing? What proportion are NIH funded. Providing a mechanism for NIH funded projects to share
- Number of journals referencing PhysioNet as recommended repository
- Analysis of emails. What is the topic? Etc.
